## We assume that you already have the data downloaded

In [ ]:
from data.dataloaders import *
from data.collate_proteins import *

dataset = FoldbenchProteinDataset(
    json_path="/content/af_subset/jsons/fb_protein.json",
    msa_root="/content/af_subset/foldbench_msas",
    cif_root="/content/af_subset/reference_structures",
    max_msa_seqs=128,
    min_identity=0.85,
    verbose=True, max_extra_msa_seqs=16,
    max_templates=4)

print("N ejemplos finales:", len(dataset))
print("Primeros descartados:", dataset.dropped[:10])

sample = dataset[0]
for k, v in sample.items():
    if torch.is_tensor(v):
        print(k, v.shape, v.dtype)
    else:
        print(k, type(v), v)

Dataset valid examples: 39
Dropped examples: 200
  query_name msa_chain_id matched_chain_id  match_identity
0       7QRJ            A                A        1.000000
1       7QRR            A                F        1.000000
2       7QUV            A                B        1.000000
3       7SPQ            A                A        1.000000
4       7TH0            A                A        0.953488
N ejemplos finales: 39
Primeros descartados: [('7QWE', 'no_msa'), ('7XPT', 'no_msa'), ('7XVQ', 'no_msa'), ('7Y37', 'no_chain_match'), ('7YLS', 'no_msa'), ('7YUL', 'no_msa'), ('7ZOP', 'no_msa'), ('8A51', 'no_msa'), ('8AMY', 'no_msa'), ('8AN5', 'no_msa')]
id <class 'str'> 7QRJ
msa_chain_id <class 'str'> A
matched_chain_id <class 'str'> A
template_chain_ids <class 'list'> ['B']
match_identity torch.Size([]) torch.float32
sequence_str <class 'str'> MSISSLLEKNIYNVHNKSNTLTNVPANPTGNTNTVWSNSNFTPPHLMYGASDITQAIGNISLTTGSFSLSLSGPWASPLVQNVAYTKINNLVNLTFPPFQANATSSAVINSAIGALPADLRPTTNIQVDFEIFVIDDGNRPVNPGLIT

## DataLoader

In [28]:
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_proteins)

batch = next(iter(loader))

for k, v in batch.items():
    if torch.is_tensor(v):
        print(k, v.shape, v.dtype)
    else:
        print(k, type(v))

id <class 'list'>
msa_chain_id <class 'list'>
matched_chain_id <class 'list'>
template_chain_ids <class 'list'>
match_identity torch.Size([2]) torch.float32
sequence_str <class 'list'>
seq_tokens torch.Size([2, 794]) torch.int64
seq_mask torch.Size([2, 794]) torch.float32
msa_tokens torch.Size([2, 128, 794]) torch.int64
msa_mask torch.Size([2, 128, 794]) torch.float32
extra_msa_feat torch.Size([2, 16, 794, 25]) torch.float32
extra_msa_mask torch.Size([2, 16, 794]) torch.float32
template_angle_feat <class 'NoneType'>
template_pair_feat <class 'NoneType'>
template_mask <class 'NoneType'>
coords_n torch.Size([2, 794, 3]) torch.float32
coords_ca torch.Size([2, 794, 3]) torch.float32
coords_c torch.Size([2, 794, 3]) torch.float32
dist_map torch.Size([2, 794, 794]) torch.float32
valid_res_mask torch.Size([2, 794]) torch.float32
valid_backbone_mask torch.Size([2, 794]) torch.float32
pair_mask torch.Size([2, 794, 794]) torch.float32
backbone_pair_mask torch.Size([2, 794, 794]) torch.float32
to

## Training The Same Model as DeepMind (this needs a LOT of GPUs): 

In [ ]:
import torch  # Tensor core
from torch.utils.data import DataLoader  # PyTorch loader

from data import FoldbenchProteinDataset, collate_proteins, AA_VOCAB  # Repo data
from model.alphafold2 import AlphaFold2  # AF2 model
from model.alphafold2_full_loss import AlphaFoldLoss  # Global loss
from training.seeds import seed_everything  # Seed helper
from training.scheduler_warmup import build_optimizer_and_scheduler  # Opt + scheduler
from training.ema import EMA  # EMA
from training.autocast import build_amp_config  # AMP
from training.train_alphafold2 import train_alphafold2  # Training loop
from training.colab_utils import is_colab  # Colab utility

# ============================================================
# Seed + device
# ============================================================
seed_everything(77, deterministic=False)  # Fix randomness

device = "cuda" if torch.cuda.is_available() else "cpu"  # Select device
print("device:", device)  # Show device

# ============================================================
# Dataset + DataLoader
#   Values close to AF2 for MSA/templates
# ============================================================
dataset = FoldbenchProteinDataset(
    manifest_csv="data/showcase_manifest.csv",  # Local manifest
    max_msa_seqs=128,  # AF2 initial training MSA clusters
    max_extra_msa_seqs=1024,  # AF2 initial training extra MSA
    max_templates=4,  # AF2 max templates
    min_identity=0.90,  # Target-chain filter
    min_template_identity=0.80,  # Local template filter
    single_sequence_mode=False,  # Use real MSA
    crop_size=64, # Crop de Length of the protein (L)
    random_crop=True, # Random Crop of the Protein
    verbose=True,  # Dataset logging
)

loader = DataLoader(
    dataset,  # AF2 dataset
    batch_size=1,  # Safe batch size
    shuffle=True,  # Shuffle samples
    collate_fn=collate_proteins,  # Custom collate
)

print("dataset len:", len(dataset))  # Dataset size
print("loader len:", len(loader))  # Loader size

# ============================================================
# Ideal local backbone geometry
#   order: N, CA, C, O
# ============================================================
ideal_backbone_local = torch.tensor([
    [-1.458,  0.000,  0.000],   # Ideal N
    [ 0.000,  0.000,  0.000],   # Ideal CA
    [ 0.547,  1.426,  0.000],   # Ideal C
    [ 0.224,  2.617,  0.000],   # Ideal O
], dtype=torch.float32)

# ============================================================
# Model
#   Canonical AF2 architecture as exposed by this repo
# ============================================================
n_tokens = max(AA_VOCAB.values()) + 1  # Input vocab size
pad_idx = AA_VOCAB["-"]  # Pad token

model = AlphaFold2(
    n_tokens=n_tokens,  # Input vocab
    c_m=256,  # AF2 MSA channel
    c_z=128,  # AF2 pair channel
    c_s=384,  # AF2 single channel
    max_relpos=32,  # AF2 RelPos
    pad_idx=pad_idx,  # Pad token
    num_evoformer_blocks=48,  # AF2 Evoformer blocks
    num_structure_blocks=8,  # AF2 Structure blocks
    transition_expansion_evoformer=4,  # Evoformer FFN expansion
    transition_expansion_structure=4,  # Structure FFN expansion
    use_block_specific_params=False,  # Shared weights AF2
    dist_bins=64,  # AF2 distogram bins
    plddt_bins=50,  # AF2 pLDDT bins
    n_torsions=3,  # Current compatibility; full AF2 uses 7
    num_res_blocks_torsion=2,  # AF2-like torsion resblocks
    recycle_min_bin=3.25,  # AF2 recycling min bin
    recycle_max_bin=20.75,  # AF2 recycling max bin
    recycle_dist_bins=15,  # AF2 recycling num bins
    extra_msa_stack_enabled=True,  # Enable extra MSA
    template_stack_enabled=True,  # Enable templates
    recycle_single_enabled=True,  # Recycle single
    evoformer_pair_stack_enabled=True,  # Enable pair stack
    evoformer_triangle_multiplication_enabled=True,  # Enable triangle multiplication
    evoformer_triangle_attention_enabled=True,  # Enable triangle attention
    evoformer_pair_transition_enabled=True,  # Enable pair transition
    recycle_pair_enabled=True,  # Recycle pair
    recycle_position_enabled=True,  # Recycle position
    extra_msa_dim=25,  # AF2 extra MSA feature dim
    extra_msa_c_e=64,  # AF2 extra MSA channel
    extra_msa_num_blocks=4,  # AF2 extra MSA blocks
    template_angle_dim=51,  # AF2 template angle dim
    template_pair_dim=88,  # AF2 template pair dim
    template_c_t=64,  # AF2 template channel
    template_num_blocks=2,  # AF2 template blocks
    structure_pair_context_enabled=True,  # Use z in structure
    distogram_head_enabled=True,  # Distogram head
    plddt_head_enabled=True,  # pLDDT head
    torsion_head_enabled=True,  # Torsion head
).to(device)

# ============================================================
# Loss
#   AF2-like weights used in this repo
# ============================================================
criterion = AlphaFoldLoss(
    fape_length_scale=10.0,  # AF2 FAPE scale
    fape_clamp_distance=10.0,  # AF2 FAPE clamp
    dist_num_bins=64,  # Distogram bins
    dist_min_bin=2.3125,  # AF2 reference min bin
    dist_max_bin=21.6875,  # AF2 reference max bin
    plddt_num_bins=50,  # pLDDT bins
    plddt_inclusion_radius=15.0,  # AF2 pLDDT radius
    w_fape=0.5,  # Final FAPE weight
    w_aux=0.5,  # Auxiliary structure weight
    w_dist=0.3,  # Distogram weight
    w_plddt=0.01,  # pLDDT weight
    w_torsion=0.01,  # Torsion weight
)

# ============================================================
# Optimizer + Scheduler
#   As close as possible with the local helper
# ============================================================
epochs = 20  # Local example
grad_accum_steps = 1  # No accumulation

steps_per_epoch = (len(loader) + grad_accum_steps - 1) // grad_accum_steps  # Steps/epoch
total_steps = epochs * steps_per_epoch  # Total steps
warmup_steps = min(1000, max(10, total_steps // 10))  # AF2-like warmup capped for local run

optimizer, scheduler = build_optimizer_and_scheduler(
    model=model,  # Trainable model
    lr=1e-3,  # AF2/OpenFold reference LR
    weight_decay=0.0,  # No explicit WD
    betas=(0.9, 0.999),  # Classic Adam betas
    eps=1e-5,  # AF2 reference eps
    total_steps=total_steps,  # Scheduler steps
    warmup_steps=warmup_steps,  # Warmup
    min_lr=1e-5,  # LR floor
)

# ============================================================
# EMA
# ============================================================
ema = EMA(
    model=model,  # Source model
    decay=0.999,  # AF2-like EMA
    device="cpu",  # EMA on CPU
    use_num_updates=True,  # Correct early steps
)

# ============================================================
# AMP config
# ============================================================
amp_cfg = build_amp_config(
    device=device,  # Current device
    amp_enabled=(device == "cuda"),  # AMP only on CUDA
    amp_dtype="bf16",  # AF2/OpenFold-like precision
)

scaler = amp_cfg["scaler"]  # Grad scaler if applicable

print("AMP cfg:", amp_cfg)  # Show AMP config

In [ ]:
# ============================================================
# Training
# ============================================================
result = train_alphafold2(
    model=model,  # AF2 model
    train_loader=loader,  # Training loader
    optimizer=optimizer,  # Optimizer
    criterion=criterion,  # Global loss
    scheduler=scheduler,  # LR scheduler
    ema=ema,  # EMA
    scaler=scaler,  # AMP scaler
    device=device,  # Training device
    epochs=epochs,  # Example epochs
    start_epoch=0,  # Starting epoch
    global_step=0,  # Starting step
    amp_enabled=amp_cfg["amp_enabled"],  # Enable AMP
    amp_dtype=amp_cfg["amp_dtype_requested"],  # AMP dtype
    grad_clip=1.0,  # Gradient clipping
    grad_accum_steps=grad_accum_steps,  # Gradient accumulation
    log_every=1,  # Logging frequency
    log_grad_norm=True,  # Log grad norm
    log_mem=True,  # Log memory
    monitor_name="loss",  # Main metric
    monitor_mode="min",  # Minimize loss
    best_metric=None,  # No previous best
    ckpt_dir="checkpoints_af2_ref",  # Checkpoint folder
    run_name="af2_reference_like",  # Run name
    save_every=1,  # Save every epoch
    save_last=True,  # Save last
    resume_path=None,  # No resume
    num_recycles=3,  # AF2 recycles
    stochastic_recycling=True,  # Stochastic recycling
    ideal_backbone_local=ideal_backbone_local.to(device),  # Ideal backbone
    drive_ckpt_dir="/content/drive/MyDrive/af2_ckpts" if is_colab() else None,  # Optional Drive
    copy_fixed_to_drive=True,  # Copy to Drive
    fixed_drive_name="latest_af2_reference_like.pt",  # Fixed Drive name
    on_oom="skip",  # Skip OOM
    config={
        "amp_dtype": amp_cfg["amp_dtype_requested"],  # AMP dtype
        "grad_clip": 1.0,  # Grad clip
        "epochs": epochs,  # Epochs
        "lr": 1e-3,  # Base LR
        "weight_decay": 0.0,  # Base WD
        "total_steps": total_steps,  # Total steps
        "warmup_steps": warmup_steps,  # Warmup
        "num_evoformer_blocks": 48,  # AF2 Evoformer
        "num_structure_blocks": 8,  # AF2 Structure
        "c_m": 256,  # MSA channel
        "c_z": 128,  # Pair channel
        "c_s": 384,  # Single channel
        "max_msa_seqs": 128,  # Max MSA
        "max_extra_msa_seqs": 1024,  # Max extra MSA
        "max_templates": 4,  # Max templates
        "num_recycles": 3,  # Recycles
    },
)

print(result)  # Final result

---

## Model For chunk of 4x NVIDIA a100 (not being tested)

In [ ]:
import torch  # Core tensor library
from torch.utils.data import DataLoader  # PyTorch dataloader

from data import FoldbenchProteinDataset, collate_proteins, AA_VOCAB  # Project data utils
from model.alphafold2 import AlphaFold2  # Main AF2 model
from model.alphafold2_full_loss import AlphaFoldLoss  # Global training loss
from training.seeds import seed_everything  # Reproducibility
from training.scheduler_warmup import build_optimizer_and_scheduler  # AdamW + scheduler
from training.ema import EMA  # Exponential moving average
from training.autocast import build_amp_config  # Mixed precision helpers
from training.train_alphafold2 import train_alphafold2  # Training loop
from training.colab_utils import is_colab  # Optional Colab helpers

# ============================================================
# Seed + device
# ============================================================
seed_everything(42, deterministic=False)  # Set random seed

device = "cuda" if torch.cuda.is_available() else "cpu"  # Pick device
print("device:", device)  # Print device

# ============================================================
# Dataset + DataLoader
#   Medium-size setup for a typical 4x A100 cluster
# ============================================================
dataset = FoldbenchProteinDataset(
    manifest_csv="data/showcase_manifest.csv",  # Replace with your full manifest if needed
    max_msa_seqs=64,  # Main MSA depth
    max_extra_msa_seqs=256,  # Extra MSA depth
    max_templates=2,  # Number of templates
    min_identity=0.90,  # Target-to-chain match threshold
    min_template_identity=0.80,  # Template match threshold
    single_sequence_mode=False,  # Use real MSA features
    verbose=True,  # Print dataset summary
)

loader = DataLoader(
    dataset,  # Dataset object
    batch_size=1,  # Per-GPU batch size if using DDP
    shuffle=True,  # Shuffle training order
    collate_fn=collate_proteins,  # Custom batch collation
)

print("dataset len:", len(dataset))  # Dataset size
print("loader len:", len(loader))  # Number of batches

# ============================================================
# Ideal local backbone geometry
#   Atom order: N, CA, C, O
# ============================================================
ideal_backbone_local = torch.tensor([
    [-1.458,  0.000,  0.000],   # Ideal N position
    [ 0.000,  0.000,  0.000],   # Ideal CA position
    [ 0.547,  1.426,  0.000],   # Ideal C position
    [ 0.224,  2.617,  0.000],   # Ideal O position
], dtype=torch.float32)

# ============================================================
# Model
#   Medium AF2-like configuration
# ============================================================
n_tokens = max(AA_VOCAB.values()) + 1  # Vocabulary size
pad_idx = AA_VOCAB["-"]  # Padding token id

model = AlphaFold2(
    n_tokens=n_tokens,  # Input vocab size
    c_m=192,  # MSA channel width
    c_z=96,  # Pair channel width
    c_s=256,  # Single channel width
    max_relpos=32,  # Relative position window
    pad_idx=pad_idx,  # Padding token
    num_evoformer_blocks=12,  # Number of Evoformer blocks
    num_structure_blocks=4,  # Number of Structure blocks
    transition_expansion_evoformer=4,  # Evoformer transition expansion
    transition_expansion_structure=4,  # Structure transition expansion
    use_block_specific_params=False,  # Share structure block weights
    dist_bins=64,  # Distogram bins
    plddt_bins=50,  # pLDDT bins
    n_torsions=3,  # Current dataset supports backbone torsions only
    num_res_blocks_torsion=2,  # Torsion head depth
    recycle_min_bin=3.25,  # Recycling distance min bin
    recycle_max_bin=20.75,  # Recycling distance max bin
    recycle_dist_bins=15,  # Recycling distance bins
    extra_msa_stack_enabled=True,  # Enable extra MSA stack
    template_stack_enabled=True,  # Enable template stack
    recycle_single_enabled=True,  # Recycle single representation
    evoformer_pair_stack_enabled=True,  # Enable pair stack
    evoformer_triangle_multiplication_enabled=True,  # Enable triangle multiplication
    evoformer_triangle_attention_enabled=True,  # Enable triangle attention
    evoformer_pair_transition_enabled=True,  # Enable pair transition
    recycle_pair_enabled=True,  # Recycle pair representation
    recycle_position_enabled=True,  # Recycle positional signal
    extra_msa_dim=25,  # Extra MSA feature dim
    extra_msa_c_e=48,  # Extra MSA hidden width
    extra_msa_num_blocks=2,  # Extra MSA stack depth
    template_angle_dim=51,  # Template angle feature dim
    template_pair_dim=88,  # Template pair feature dim
    template_c_t=48,  # Template hidden width
    template_num_blocks=2,  # Template stack depth
    structure_pair_context_enabled=True,  # Feed pair context into structure
    distogram_head_enabled=True,  # Enable distogram head
    plddt_head_enabled=True,  # Enable pLDDT head
    torsion_head_enabled=True,  # Enable torsion head
).to(device)

# ============================================================
# Loss
# ============================================================
criterion = AlphaFoldLoss(
    fape_length_scale=10.0,  # FAPE length scale
    fape_clamp_distance=10.0,  # FAPE clamp distance
    dist_num_bins=64,  # Distogram bins
    dist_min_bin=2.3125,  # Distogram min bin
    dist_max_bin=21.6875,  # Distogram max bin
    plddt_num_bins=50,  # pLDDT bins
    plddt_inclusion_radius=15.0,  # pLDDT supervision radius
    w_fape=0.5,  # Final structure weight
    w_aux=0.5,  # Auxiliary structure weight
    w_dist=0.3,  # Distogram weight
    w_plddt=0.01,  # pLDDT weight
    w_torsion=0.01,  # Torsion weight
)

# ============================================================
# Optimizer + Scheduler
# ============================================================
epochs = 30  # Number of epochs
grad_accum_steps = 1  # Gradient accumulation steps

steps_per_epoch = (len(loader) + grad_accum_steps - 1) // grad_accum_steps  # Steps per epoch
total_steps = epochs * steps_per_epoch  # Total optimizer steps
warmup_steps = min(1000, max(20, total_steps // 10))  # Warmup schedule

optimizer, scheduler = build_optimizer_and_scheduler(
    model=model,  # Trainable model
    lr=3e-4,  # Medium-scale learning rate
    weight_decay=0.0,  # No explicit weight decay
    betas=(0.9, 0.999),  # Adam betas
    eps=1e-5,  # Adam epsilon
    total_steps=total_steps,  # Scheduler total steps
    warmup_steps=warmup_steps,  # Scheduler warmup
    min_lr=1e-5,  # Minimum learning rate
)

# ============================================================
# EMA
# ============================================================
ema = EMA(
    model=model,  # Source model
    decay=0.999,  # EMA decay
    device="cpu",  # Keep EMA weights on CPU
    use_num_updates=True,  # Bias-correct early steps
)

# ============================================================
# AMP config
# ============================================================
amp_cfg = build_amp_config(
    device=device,  # Active device
    amp_enabled=(device == "cuda"),  # Enable AMP on CUDA
    amp_dtype="bf16",  # Use bf16 when possible
)

scaler = amp_cfg["scaler"]  # Grad scaler if needed
print("AMP cfg:", amp_cfg)  # Print AMP setup



In [ ]:
# ============================================================
# Train
# ============================================================
result = train_alphafold2(
    model=model,  # AF2 model
    train_loader=loader,  # Training loader
    optimizer=optimizer,  # Optimizer
    criterion=criterion,  # Loss function
    scheduler=scheduler,  # LR scheduler
    ema=ema,  # EMA tracker
    scaler=scaler,  # AMP scaler
    device=device,  # Training device
    epochs=epochs,  # Number of epochs
    start_epoch=0,  # Initial epoch index
    global_step=0,  # Initial global step
    amp_enabled=amp_cfg["amp_enabled"],  # AMP flag
    amp_dtype=amp_cfg["amp_dtype_requested"],  # AMP dtype
    grad_clip=1.0,  # Gradient clipping
    grad_accum_steps=grad_accum_steps,  # Gradient accumulation
    log_every=1,  # Logging frequency
    log_grad_norm=True,  # Log gradient norm
    log_mem=True,  # Log memory usage
    monitor_name="loss",  # Main checkpoint metric
    monitor_mode="min",  # Lower is better
    best_metric=None,  # No previous best
    ckpt_dir="checkpoints_af2_medium",  # Checkpoint folder
    run_name="af2_medium_cluster",  # Run name
    save_every=1,  # Save every epoch
    save_last=True,  # Save last checkpoint
    resume_path=None,  # No resume checkpoint
    num_recycles=1,  # Training recycles
    stochastic_recycling=True,  # Randomize recycle count
    ideal_backbone_local=ideal_backbone_local.to(device),  # Ideal backbone template
    drive_ckpt_dir="/content/drive/MyDrive/af2_ckpts" if is_colab() else None,  # Optional Colab backup
    copy_fixed_to_drive=True,  # Copy latest fixed checkpoint
    fixed_drive_name="latest_af2_medium.pt",  # Fixed backup filename
    on_oom="skip",  # Skip batch on OOM
    config={
        "amp_dtype": amp_cfg["amp_dtype_requested"],  # AMP dtype
        "grad_clip": 1.0,  # Gradient clip value
        "epochs": epochs,  # Epoch count
        "lr": 3e-4,  # Learning rate
        "weight_decay": 0.0,  # Weight decay
        "total_steps": total_steps,  # Total steps
        "warmup_steps": warmup_steps,  # Warmup steps
        "num_evoformer_blocks": 12,  # Evoformer depth
        "num_structure_blocks": 4,  # Structure depth
        "c_m": 192,  # MSA width
        "c_z": 96,  # Pair width
        "c_s": 256,  # Single width
        "max_msa_seqs": 64,  # Main MSA depth
        "max_extra_msa_seqs": 256,  # Extra MSA depth
        "max_templates": 2,  # Template count
        "num_recycles": 1,  # Recycle count
    },
)

print(result)  # Final training result